In [ ]:
# import kagglehub
# 
# # Download latest version
# path = kagglehub.dataset_download("seryouxblaster764/fgvc-aircraft")
# 
# print("Path to dataset files:", path)

In [19]:
from torch.utils.data import Dataset
from PIL import Image
import os
import pandas as pd
from torchvision.models import resnet18


class AircraftDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        # Создаем отображение классов в индексы
        self.class_to_idx = {cls: idx for idx, cls in enumerate(sorted(self.data['Classes'].unique()))}
        self.data['label'] = self.data['Classes'].map(self.class_to_idx)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        label = row['label']
        if self.transform:
            image = self.transform(image)
        return image, label


In [20]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


train_dataset = AircraftDataset('data/fgvc-aircraft/train.csv', 'data/fgvc-aircraft/images', transform)
val_dataset = AircraftDataset('data/fgvc-aircraft/val.csv', 'data/fgvc-aircraft/images', transform)
test_dataset = AircraftDataset('data/fgvc-aircraft/test.csv', 'data/fgvc-aircraft/images', transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AircraftCNN(nn.Module):
    def __init__(self, num_classes):
        super(AircraftCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))  # гарантированный выход 4×4

        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # -> [B, 32, H/2, W/2]
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # -> [B, 64, H/4, W/4]
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # -> [B, 128, H/8, W/8]
        x = self.adaptive_pool(x)  # -> [B, 128, 4, 4]
        x = x.view(x.size(0), -1)  # -> [B, 2048]
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


In [22]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total


In [23]:
import torch
from torch import nn, optim
from tqdm import tqdm
import wandb


def train_model(model, train_loader, val_loader, device, epochs=10):
    wandb.init(project="aircraft-classifier",
               config={"epochs": epochs,
                       "lr": 1e-4,
                       "batch_size": train_loader.batch_size}
               )
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0.0

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total
        val_acc = evaluate_model(model, val_loader, device)

        wandb.log({
            "train_loss": running_loss / total,
            "train_acc": train_acc,
            "val_acc": val_acc,
            "epoch": epoch + 1
        })

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print("✅ Model saved.")
            wandb.run.summary["best_val_acc"] = best_acc
            
        scheduler.step()
        
        print(f"Epoch {epoch + 1}: Loss={running_loss / total:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")


In [24]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = resnet18(weights='IMAGENET1K_V1')  # предобученные веса
model.fc = nn.Linear(model.fc.in_features, 100)  # подгон под твои классы
model = model.to(device)

train_model(model, train_loader, val_loader, device, epochs=20)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Школа Рока/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:04<00:00, 9.58MB/s]


epoch,▁▂▂▃▃▄▅▅▆▆▇▇█
train_acc,▁▂▂▄▄▅▅▅▆▇▇██
train_loss,█▇▆▅▄▄▃▃▂▂▁▁▁
val_acc,▁▂▃▄▅▆▆▆▆▇███
best_val_acc,0.07051
epoch,13
train_acc,0.09088
train_loss,4.04563
val_acc,0.06961


Epoch 1/20: 100%|██████████| 105/105 [01:13<00:00,  1.44it/s]


✅ Model saved.
Epoch 1: Loss=4.2328, Train Acc=0.0981, Val Acc=0.2145


Epoch 2/20: 100%|██████████| 105/105 [01:12<00:00,  1.46it/s]


...

Epoch 18: Loss=0.1379, Train Acc=0.9985, Val Acc=0.6991


Epoch 19/20: 100%|██████████| 105/105 [01:12<00:00,  1.45it/s]


Epoch 19: Loss=0.1269, Train Acc=0.9988, Val Acc=0.7006


Epoch 20/20: 100%|██████████| 105/105 [01:12<00:00,  1.45it/s]


Epoch 20: Loss=0.1227, Train Acc=0.9988, Val Acc=0.7021
